In [1]:

import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

In [14]:
# load fully corrected datasets
from rdma.utils.data import read_json_file, print_json_structure


# all human labels
human_corrections_full = read_json_file("data/dataset/rare_disease_corrections_john.json")
print("-------- Human Corrections File ------")
print_json_structure(human_corrections_full)
# human + supervisor labels
human_rdma_corrections = read_json_file("data/dataset/adam_corrections_v2.json")
# supervisor labels
print("------- Supervisor Corrections File -------")
rdma_corrections = read_json_file("data/results/supervisor/multistage_no_min.json")
print_json_structure(rdma_corrections)

-------- Human Corrections File ------
Dictionary:
  metadata (dict): 
  Dictionary:
    timestamp (str): 
    total_entities_in_file (int): 
    reviewed_entities (int): 
  corrected_annotations (list): 
  List: (333 items)
    Item 0 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 1 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 2 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 3 (dict): 
    Dictionary:
      entity (str): 
      document_id (str): 
      orpha_code (str): 
      category (str): 
      is_rare_disease (bool): 
      ... and 2 more items
    Item 4 (dict): 

# Comparing RDMA (only) and my annotations, because they go through the entire set of possible annotations

In [15]:
"""
Set-Based Rare Disease Annotator Agreement Analysis

This script computes the inter-annotator agreement between human corrections
and RDMA supervisor corrections for rare disease entity recognition
using a set-based approach at the document level.
"""

import json
import pandas as pd
import numpy as np
from typing import Dict, List, Set, Tuple, Any, Optional
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import cohen_kappa_score
import scipy.stats as stats
import os

def read_json_file(filename: str) -> dict:
    """Read a JSON file and return its contents."""
    with open(filename, 'r') as f:
        return json.load(f)

def extract_document_entity_sets(human_corrections: dict, supervisor_corrections: dict) -> Tuple[Dict[str, Set[str]], Dict[str, Set[str]]]:
    """
    Extract document-level entity sets from both annotations.
    
    Args:
        human_corrections: Human corrections dictionary
        supervisor_corrections: Supervisor corrections dictionary
        
    Returns:
        Tuple of (human_doc_entities, supervisor_doc_entities)
        Where each is a dictionary mapping document_id to a set of entities marked as rare diseases
    """
    # Initialize dictionaries to store entities by document
    human_doc_entities = defaultdict(set)
    supervisor_doc_entities = defaultdict(set)
    
    # Extract human annotations
    if human_corrections and 'corrected_annotations' in human_corrections:
        for annotation in human_corrections['corrected_annotations']:
            if ('entity' in annotation and 
                'document_id' in annotation and 
                'is_rare_disease' in annotation):
                
                doc_id = annotation['document_id']
                entity = annotation['entity'].lower().strip()  # Normalize for comparison
                
                # Only add entities marked as rare diseases
                if annotation['is_rare_disease']:
                    human_doc_entities[doc_id].add(entity)
    
    # Extract supervisor annotations - only from true positives and false negatives
    if supervisor_corrections and 'results' in supervisor_corrections:
        # Process true positives and false negatives (entities that should be rare diseases)
        for category in ['true_positives', 'false_negatives']:
            if category in supervisor_corrections['results']:
                for annotation in supervisor_corrections['results'][category]:
                    if ('entity' in annotation and 
                        'document_id' in annotation and 
                        'is_rare_disease' in annotation):
                        
                        doc_id = annotation['document_id']
                        entity = annotation['entity'].lower().strip()  # Normalize for comparison
                        
                        # Only add entities marked as rare diseases
                        if annotation['is_rare_disease']:
                            supervisor_doc_entities[doc_id].add(entity)
    
    # Get all documents from both sets to ensure we have complete coverage
    all_docs = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Ensure each document is represented in both dictionaries, even if empty
    for doc_id in all_docs:
        if doc_id not in human_doc_entities:
            human_doc_entities[doc_id] = set()
        if doc_id not in supervisor_doc_entities:
            supervisor_doc_entities[doc_id] = set()
    
    return human_doc_entities, supervisor_doc_entities

def compute_agreement_metrics(human_doc_entities: Dict[str, Set[str]], 
                              supervisor_doc_entities: Dict[str, Set[str]],
                              excluded_docs: Optional[List[str]] = None) -> Dict[str, Any]:
    """
    Compute agreement metrics between human and supervisor annotations.
    
    Args:
        human_doc_entities: Dictionary mapping document_id to set of human-annotated rare disease entities
        supervisor_doc_entities: Dictionary mapping document_id to set of supervisor-annotated rare disease entities
        excluded_docs: Optional list of document IDs to exclude from evaluation
        
    Returns:
        Dictionary with agreement metrics
    """
    # Initialize counters for overall metrics
    total_human_entities = 0
    total_supervisor_entities = 0
    total_common_entities = 0
    
    # Initialize document-level metrics
    docs_with_entities = set()
    doc_metrics = {}
    
    # Get all document IDs
    all_doc_ids = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Remove excluded documents if specified
    if excluded_docs:
        all_doc_ids = all_doc_ids - set(excluded_docs)
    
    # Collect all unique entities across all documents
    all_unique_entities = set()
    for doc_id in all_doc_ids:
        all_unique_entities.update(human_doc_entities[doc_id])
        all_unique_entities.update(supervisor_doc_entities[doc_id])
    
    # For Cohen's Kappa calculation
    all_entities_list = sorted(list(all_unique_entities))
    entity_to_idx = {entity: idx for idx, entity in enumerate(all_entities_list)}
    
    # Lists to store human and supervisor ratings for each entity in each document
    human_ratings = []
    supervisor_ratings = []
    
    # Process each document
    for doc_id in all_doc_ids:
        human_entities = human_doc_entities.get(doc_id, set())
        supervisor_entities = supervisor_doc_entities.get(doc_id, set())
        
        # Count entities for this document
        human_count = len(human_entities)
        supervisor_count = len(supervisor_entities)
        
        # Calculate intersection (common entities)
        common_entities = human_entities & supervisor_entities
        common_count = len(common_entities)
        
        # For each entity in this document, record human and supervisor ratings (1 = yes, 0 = no)
        doc_all_entities = human_entities | supervisor_entities
        
        # Only consider documents with at least one entity from either annotator
        if doc_all_entities:
            docs_with_entities.add(doc_id)
            
            for entity in doc_all_entities:
                human_ratings.append(1 if entity in human_entities else 0)
                supervisor_ratings.append(1 if entity in supervisor_entities else 0)
                
        # Calculate unique entities
        human_only_count = len(human_entities - supervisor_entities)
        supervisor_only_count = len(supervisor_entities - human_entities)
        
        # Calculate agreement metrics for this document
        if human_count > 0 or supervisor_count > 0:
            # Calculate precision, recall, F1
            precision = common_count / supervisor_count if supervisor_count > 0 else 0
            recall = common_count / human_count if human_count > 0 else 0
            f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            # Calculate percent agreement
            total_judgments = len(human_entities | supervisor_entities)
            agreements = common_count + (total_judgments - (human_count + supervisor_count - common_count))
            percent_agreement = agreements / total_judgments if total_judgments > 0 else 0
            
            # Store document metrics
            doc_metrics[doc_id] = {
                'human_entities': human_entities,
                'supervisor_entities': supervisor_entities,
                'common_entities': common_entities,
                'human_count': human_count,
                'supervisor_count': supervisor_count,
                'common_count': common_count,
                'human_only_count': human_only_count,
                'supervisor_only_count': supervisor_only_count,
                'precision': precision,
                'recall': recall,
                'f1_score': f1_score,
                'percent_agreement': percent_agreement
            }
            
            # Update overall counts
            total_human_entities += human_count
            total_supervisor_entities += supervisor_count
            total_common_entities += common_count
    
    # Calculate overall metrics
    overall_precision = total_common_entities / total_supervisor_entities if total_supervisor_entities > 0 else 0
    overall_recall = total_common_entities / total_human_entities if total_human_entities > 0 else 0
    overall_f1 = 2 * overall_precision * overall_recall / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0
    
    # Calculate Cohen's Kappa
    if human_ratings and supervisor_ratings:
        kappa = cohen_kappa_score(human_ratings, supervisor_ratings)
    else:
        kappa = 0
    
    # Calculate Pearson correlation
    if human_ratings and supervisor_ratings and len(human_ratings) > 1:
        pearson_corr, p_value = stats.pearsonr(human_ratings, supervisor_ratings)
    else:
        pearson_corr = 0
        p_value = 1
    
    # Calculate average document-level metrics (only for documents with entities)
    docs_with_metrics = [m for doc_id, m in doc_metrics.items() if doc_id in docs_with_entities]
    
    if docs_with_metrics:
        avg_precision = np.mean([m['precision'] for m in docs_with_metrics])
        avg_recall = np.mean([m['recall'] for m in docs_with_metrics])
        avg_f1 = np.mean([m['f1_score'] for m in docs_with_metrics])
        avg_percent_agreement = np.mean([m['percent_agreement'] for m in docs_with_metrics])
        
        # Calculate standard deviations
        std_precision = np.std([m['precision'] for m in docs_with_metrics])
        std_recall = np.std([m['recall'] for m in docs_with_metrics])
        std_f1 = np.std([m['f1_score'] for m in docs_with_metrics])
        std_percent_agreement = np.std([m['percent_agreement'] for m in docs_with_metrics])
    else:
        avg_precision = avg_recall = avg_f1 = avg_percent_agreement = 0
        std_precision = std_recall = std_f1 = std_percent_agreement = 0
    
    return {
        'total_documents': len(docs_with_entities),
        'total_unique_entities': len(all_unique_entities),
        'total_human_entities': total_human_entities,
        'total_supervisor_entities': total_supervisor_entities,
        'total_common_entities': total_common_entities,
        'total_human_only': total_human_entities - total_common_entities,
        'total_supervisor_only': total_supervisor_entities - total_common_entities,
        'overall_precision': overall_precision,
        'overall_recall': overall_recall,
        'overall_f1': overall_f1,
        'cohen_kappa': kappa,
        'pearson_correlation': pearson_corr,
        'pearson_p_value': p_value,
        'avg_precision': avg_precision,
        'avg_recall': avg_recall,
        'avg_f1': avg_f1,
        'avg_percent_agreement': avg_percent_agreement,
        'std_precision': std_precision,
        'std_recall': std_recall,
        'std_f1': std_f1,
        'std_percent_agreement': std_percent_agreement,
        'document_metrics': doc_metrics
    }

def analyze_entity_disagreements(human_doc_entities: Dict[str, Set[str]], 
                                supervisor_doc_entities: Dict[str, Set[str]]) -> Dict[str, Any]:
    """
    Analyze disagreements in entity recognition between human and supervisor.
    
    Args:
        human_doc_entities: Dictionary mapping document_id to set of human-annotated rare disease entities
        supervisor_doc_entities: Dictionary mapping document_id to set of supervisor-annotated rare disease entities
        
    Returns:
        Dictionary with disagreement analysis
    """
    # Get all document IDs
    all_doc_ids = set(human_doc_entities.keys()) | set(supervisor_doc_entities.keys())
    
    # Initialize disagreement lists
    human_only_entities = []
    supervisor_only_entities = []
    
    # Process each document
    for doc_id in all_doc_ids:
        human_entities = human_doc_entities.get(doc_id, set())
        supervisor_entities = supervisor_doc_entities.get(doc_id, set())
        
        # Find entities that are in human annotations but not in supervisor
        for entity in human_entities - supervisor_entities:
            human_only_entities.append({
                'entity': entity,
                'document_id': doc_id
            })
        
        # Find entities that are in supervisor annotations but not in human
        for entity in supervisor_entities - human_entities:
            supervisor_only_entities.append({
                'entity': entity,
                'document_id': doc_id
            })
    
    # Analyze human-only entities
    human_only_by_frequency = {}
    for item in human_only_entities:
        entity = item['entity']
        if entity not in human_only_by_frequency:
            human_only_by_frequency[entity] = 0
        human_only_by_frequency[entity] += 1
    
    # Analyze supervisor-only entities
    supervisor_only_by_frequency = {}
    for item in supervisor_only_entities:
        entity = item['entity']
        if entity not in supervisor_only_by_frequency:
            supervisor_only_by_frequency[entity] = 0
        supervisor_only_by_frequency[entity] += 1
    
    # Sort by frequency
    human_only_sorted = sorted(human_only_by_frequency.items(), key=lambda x: x[1], reverse=True)
    supervisor_only_sorted = sorted(supervisor_only_by_frequency.items(), key=lambda x: x[1], reverse=True)
    
    return {
        'human_only_entities': human_only_entities,
        'supervisor_only_entities': supervisor_only_entities,
        'human_only_by_frequency': human_only_sorted,
        'supervisor_only_by_frequency': supervisor_only_sorted,
        'total_human_only': len(human_only_entities),
        'total_supervisor_only': len(supervisor_only_entities),
        'unique_human_only': len(human_only_by_frequency),
        'unique_supervisor_only': len(supervisor_only_by_frequency)
    }

def create_agreement_visualizations(metrics: Dict[str, Any], disagreements: Dict[str, Any]) -> Dict[str, plt.Figure]:
    """
    Create visualizations for agreement analysis.
    
    Args:
        metrics: Dictionary with agreement metrics
        disagreements: Dictionary with disagreement analysis
        
    Returns:
        Dictionary mapping visualization names to figure objects
    """
    figures = {}
    
    # Set consistent color scheme
    colors = {
        'human_only': '#ff9999',  # Light red
        'common': '#66b3ff',      # Light blue
        'supervisor_only': '#99ff99',  # Light green
        'grid': '#dddddd',        # Light gray
        'highlight': '#ff6600'    # Orange for highlights
    }
    
    # 1. Overall entity counts
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    counts = [
        metrics['total_human_only'],
        metrics['total_common_entities'],
        metrics['total_supervisor_only']
    ]
    labels = ['Human Only', 'Common', 'Supervisor Only']
    bar_colors = [colors['human_only'], colors['common'], colors['supervisor_only']]
    ax1.bar(labels, counts, color=bar_colors)
    ax1.set_title('Entity Recognition Comparison', fontsize=14)
    ax1.set_ylabel('Number of Entities', fontsize=12)
    
    # Add count labels
    for i, count in enumerate(counts):
        ax1.annotate(str(count), xy=(i, count), ha='center', va='bottom', fontsize=11)
    
    ax1.grid(axis='y', linestyle='--', alpha=0.7, color=colors['grid'])
    
    # Add agreement metrics as text
    agreement_text = (
        f"Cohen's Kappa: {metrics['cohen_kappa']:.3f}\n"
        f"Pearson Correlation: {metrics['pearson_correlation']:.3f}\n"
        f"Overall F1: {metrics['overall_f1']:.3f}\n"
        f"Precision: {metrics['overall_precision']:.3f}\n"
        f"Recall: {metrics['overall_recall']:.3f}"
    )
    ax1.text(0.02, 0.97, agreement_text, transform=ax1.transAxes,
            verticalalignment='top', bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.8})
    
    plt.tight_layout()
    figures['entity_counts'] = fig1
    
    # 2. Document-level F1 scores histogram
    fig2, ax2 = plt.subplots(figsize=(10, 6))
    f1_scores = [metrics['document_metrics'][doc]['f1_score'] for doc in metrics['document_metrics'] 
                if len(metrics['document_metrics'][doc]['human_entities'] | 
                      metrics['document_metrics'][doc]['supervisor_entities']) > 0]
    
    if f1_scores:
        bins = np.linspace(0, 1, 11)  # 10 bins from 0 to 1
        ax2.hist(f1_scores, bins=bins, alpha=0.7, color=colors['common'])
        ax2.set_title('Distribution of Document-Level F1 Scores', fontsize=14)
        ax2.set_xlabel('F1 Score', fontsize=12)
        ax2.set_ylabel('Number of Documents', fontsize=12)
        ax2.grid(True, linestyle='--', alpha=0.7, color=colors['grid'])
        
        # Add mean line
        ax2.axvline(metrics['avg_f1'], color=colors['highlight'], linestyle='dashed', linewidth=2, 
                   label=f'Mean F1: {metrics["avg_f1"]:.3f}')
        ax2.legend(fontsize=11)
        
        # Set x-axis ticks
        ax2.set_xticks(np.arange(0, 1.1, 0.1))
    else:
        ax2.text(0.5, 0.5, "No documents with F1 scores", ha='center', va='center', fontsize=14)
    
    plt.tight_layout()
    figures['f1_distribution'] = fig2
    
    # 3. Top disagreements by frequency
    fig3, ax3 = plt.subplots(figsize=(12, 8))
    
    # Combine for comparison
    entity_counts = {}
    
    # Add human-only frequencies (as negative values for left side)
    for entity, freq in disagreements['human_only_by_frequency'][:10]:  # Top 10
        entity_counts[entity] = -freq
    
    # Add supervisor-only frequencies (as positive values for right side)
    for entity, freq in disagreements['supervisor_only_by_frequency'][:10]:  # Top 10
        if entity in entity_counts:
            # If entity exists in both sides, subtract supervisor freq (it's already negative)
            entity_counts[entity] = entity_counts[entity] - freq
        else:
            entity_counts[entity] = freq
    
    # Sort for display
    sorted_entities = sorted(entity_counts.items(), key=lambda x: abs(x[1]), reverse=True)[:15]  # Top 15 overall
    
    if sorted_entities:
        entities = [item[0] for item in sorted_entities]
        counts = [item[1] for item in sorted_entities]
        
        # Create colors based on which side the bar is on
        bar_colors = [colors['human_only'] if count < 0 else colors['supervisor_only'] for count in counts]
        
        y_pos = np.arange(len(entities))
        ax3.barh(y_pos, counts, color=bar_colors)
        ax3.set_yticks(y_pos)
        ax3.set_yticklabels(entities, fontsize=10)
        ax3.set_xlabel('Count (Human-only ← | → Supervisor-only)', fontsize=12)
        ax3.set_title('Top Entity Disagreements by Frequency', fontsize=14)
        
        # Add zero line
        ax3.axvline(x=0, color='black', linestyle='-', alpha=0.7)
    else:
        ax3.text(0.5, 0.5, "No disagreements to display", ha='center', va='center', fontsize=14)
    
    plt.tight_layout()
    figures['top_disagreements'] = fig3
    
    # 4. Precision-Recall by document scatterplot
    fig4, ax4 = plt.subplots(figsize=(10, 8))
    
    precisions = []
    recalls = []
    f1_scores = []
    doc_ids = []
    
    for doc_id, doc_metric in metrics['document_metrics'].items():
        if (doc_metric['human_count'] > 0 or doc_metric['supervisor_count'] > 0) and (
            doc_metric['human_count'] + doc_metric['supervisor_count'] > 0):
            precisions.append(doc_metric['precision'])
            recalls.append(doc_metric['recall'])
            f1_scores.append(doc_metric['f1_score'])
            doc_ids.append(doc_id)
    
    if precisions and recalls:
        # Create scatter plot
        scatter = ax4.scatter(recalls, precisions, c=f1_scores, cmap='viridis', 
                             alpha=0.7, s=100, edgecolors='black', linewidths=1)
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax4)
        cbar.set_label('F1 Score', fontsize=12)
        
        # Add mean lines
        ax4.axhline(y=metrics['avg_precision'], color='red', linestyle='--', 
                   alpha=0.5, label=f'Avg Precision: {metrics["avg_precision"]:.3f}')
        ax4.axvline(x=metrics['avg_recall'], color='blue', linestyle='--', 
                   alpha=0.5, label=f'Avg Recall: {metrics["avg_recall"]:.3f}')
        
        # Add F1 score contours
        f1_levels = [0.2, 0.4, 0.6, 0.8, 0.9]
        x = np.linspace(0.01, 1, 100)
        for f1 in f1_levels:
            # Formula: recall = f1 * precision / (2 * precision - f1)
            y = []
            for recall in x:
                if 2 * recall - f1 <= 0:
                    y.append(None)
                else:
                    precision = (f1 * recall) / (2 * recall - f1)
                    if 0 <= precision <= 1:
                        y.append(precision)
                    else:
                        y.append(None)
            
            # Filter out None values
            valid_points = [(xx, yy) for xx, yy in zip(x, y) if yy is not None]
            if valid_points:
                valid_x, valid_y = zip(*valid_points)
                ax4.plot(valid_x, valid_y, '--', color='gray', alpha=0.5, label=f'F1={f1}')
        
        ax4.set_xlim(0, 1.05)
        ax4.set_ylim(0, 1.05)
        ax4.set_xlabel('Recall', fontsize=12)
        ax4.set_ylabel('Precision', fontsize=12)
        ax4.set_title('Precision-Recall by Document', fontsize=14)
        ax4.grid(True, linestyle='--', alpha=0.3, color=colors['grid'])
        
        # Create a reduced legend (remove F1 contour labels)
        handles, labels = ax4.get_legend_handles_labels()
        main_handles = handles[:2]  # Just keep the first two items (avg precision and recall)
        main_labels = labels[:2]
        ax4.legend(main_handles, main_labels, loc='lower left', fontsize=10)
    else:
        ax4.text(0.5, 0.5, "Insufficient data for precision-recall plot", 
                ha='center', va='center', fontsize=14)
    
    plt.tight_layout()
    figures['precision_recall'] = fig4
    
    # 5. Document-level agreement metrics
    fig5, ax5 = plt.subplots(figsize=(10, 6))
    
    # Calculate data to plot
    doc_metrics = metrics['document_metrics']
    docs_with_entities = [doc_id for doc_id in doc_metrics 
                         if doc_metrics[doc_id]['human_count'] > 0 or 
                         doc_metrics[doc_id]['supervisor_count'] > 0]
    
    if docs_with_entities:
        # Sort by F1 score for better visualization
        docs_with_entities.sort(key=lambda x: doc_metrics[x]['f1_score'])
        
        # Extract metrics for plotting
        f1_vals = [doc_metrics[d]['f1_score'] for d in docs_with_entities]
        precision_vals = [doc_metrics[d]['precision'] for d in docs_with_entities]
        recall_vals = [doc_metrics[d]['recall'] for d in docs_with_entities]
        percent_agreement_vals = [doc_metrics[d]['percent_agreement'] for d in docs_with_entities]
        
        # Create x-axis positions
        x = np.arange(len(docs_with_entities))
        
        # Plot lines
        ax5.plot(x, f1_vals, 'o-', label='F1 Score', color='#3366cc', linewidth=2)
        ax5.plot(x, precision_vals, 'o-', label='Precision', color='#dc3912', linewidth=2)
        ax5.plot(x, recall_vals, 'o-', label='Recall', color='#ff9900', linewidth=2)
        ax5.plot(x, percent_agreement_vals, 'o-', label='% Agreement', color='#109618', linewidth=2)
        
        # Configure plot
        ax5.set_title('Document-Level Agreement Metrics', fontsize=14)
        ax5.set_xlabel('Documents (sorted by F1 score)', fontsize=12)
        ax5.set_ylabel('Metric Value', fontsize=12)
        ax5.set_ylim(0, 1.05)
        ax5.grid(True, linestyle='--', alpha=0.3, color=colors['grid'])
        ax5.legend(fontsize=10)
        
        # Remove x-tick labels (too many documents)
        ax5.set_xticks([])
        
        # Add overall metrics as text
        overall_text = (
            f"Cohen's Kappa: {metrics['cohen_kappa']:.3f}\n"
            f"Avg F1: {metrics['avg_f1']:.3f} ± {metrics['std_f1']:.3f}\n"
            f"Avg Precision: {metrics['avg_precision']:.3f} ± {metrics['std_precision']:.3f}\n"
            f"Avg Recall: {metrics['avg_recall']:.3f} ± {metrics['std_recall']:.3f}\n"
            f"Avg % Agreement: {metrics['avg_percent_agreement']:.3f} ± {metrics['std_percent_agreement']:.3f}"
        )
        ax5.text(0.02, 0.03, overall_text, transform=ax5.transAxes,
                verticalalignment='bottom', bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.8})
    else:
        ax5.text(0.5, 0.5, "No documents with agreement metrics", 
                ha='center', va='center', fontsize=14)
    
    plt.tight_layout()
    figures['document_metrics'] = fig5
    
    return figures

def print_agreement_report(metrics: Dict[str, Any], disagreements: Dict[str, Any]) -> None:
    """
    Print a detailed report of the annotator agreement analysis.
    
    Args:
        metrics: Dictionary with agreement metrics
        disagreements: Dictionary with disagreement analysis
    """
    print("=" * 80)
    print("RARE DISEASE ANNOTATOR AGREEMENT ANALYSIS REPORT")
    print("=" * 80)
    
    print("\n--- OVERALL METRICS ---")
    print(f"Documents analyzed: {metrics['total_documents']}")
    print(f"Total unique entities: {metrics['total_unique_entities']}")
    print(f"Human-annotated entities: {metrics['total_human_entities']}")
    print(f"Supervisor-annotated entities: {metrics['total_supervisor_entities']}")
    print(f"Common entities: {metrics['total_common_entities']}")
    
    print("\n--- AGREEMENT SCORES ---")
    print(f"Cohen's Kappa: {metrics['cohen_kappa']:.4f}")
    print(f"Pearson Correlation: {metrics['pearson_correlation']:.4f} (p={metrics['pearson_p_value']:.6f})")
    print(f"Overall Precision: {metrics['overall_precision']:.4f}")
    print(f"Overall Recall: {metrics['overall_recall']:.4f}")
    print(f"Overall F1 Score: {metrics['overall_f1']:.4f}")
    
    print("\nDocument-level metrics (mean ± std):")
    print(f"  Precision: {metrics['avg_precision']:.4f} ± {metrics['std_precision']:.4f}")
    print(f"  Recall: {metrics['avg_recall']:.4f} ± {metrics['std_recall']:.4f}")
    print(f"  F1 Score: {metrics['avg_f1']:.4f} ± {metrics['std_f1']:.4f}")
    print(f"  Percent Agreement: {metrics['avg_percent_agreement']:.4f} ± {metrics['std_percent_agreement']:.4f}")
    
    print("\n--- DISAGREEMENT ANALYSIS ---")
    print(f"Human-only entities: {disagreements['total_human_only']} ({disagreements['unique_human_only']} unique)")
    print(f"Supervisor-only entities: {disagreements['total_supervisor_only']} ({disagreements['unique_supervisor_only']} unique)")
    
    print("\nTop Human-only Entities (Human annotated as rare disease, Supervisor did not):")
    for entity, count in disagreements['human_only_by_frequency'][:10]:
        print(f"  {entity}: {count} occurrences")
    
    print("\nTop Supervisor-only Entities (Supervisor annotated as rare disease, Human did not):")
    for entity, count in disagreements['supervisor_only_by_frequency'][:10]:
        print(f"  {entity}: {count} occurrences")
        
    # Document Analysis
    print("\n--- DOCUMENT ANALYSIS ---")
    # Sort documents by F1 score
    sorted_docs = sorted(
        metrics['document_metrics'].items(), 
        key=lambda x: x[1].get('f1_score', 0)
    )
    
    # Only display documents that have entities
    docs_with_entities = [(doc_id, doc_metrics) for doc_id, doc_metrics in sorted_docs
                         if doc_metrics.get('human_count', 0) > 0 or doc_metrics.get('supervisor_count', 0) > 0]
    
    if docs_with_entities:
        print("\nWorst 5 Documents by F1 Score:")
        for doc_id, doc_metrics in docs_with_entities[:5]:
            print(f"  Document {doc_id}: F1={doc_metrics.get('f1_score', 0):.3f}, "
                  f"Precision={doc_metrics.get('precision', 0):.3f}, "
                  f"Recall={doc_metrics.get('recall', 0):.3f}")
            print(f"    Human entities: {len(doc_metrics.get('human_entities', set()))}, "
                  f"Supervisor entities: {len(doc_metrics.get('supervisor_entities', set()))}, "
                  f"Common: {len(doc_metrics.get('common_entities', set()))}")
        
        print("\nBest 5 Documents by F1 Score:")
        for doc_id, doc_metrics in docs_with_entities[-5:]:
            print(f"  Document {doc_id}: F1={doc_metrics.get('f1_score', 0):.3f}, "
                  f"Precision={doc_metrics.get('precision', 0):.3f}, "
                  f"Recall={doc_metrics.get('recall', 0):.3f}")
            print(f"    Human entities: {len(doc_metrics.get('human_entities', set()))}, "
                  f"Supervisor entities: {len(doc_metrics.get('supervisor_entities', set()))}, "
                  f"Common: {len(doc_metrics.get('common_entities', set()))}")
            
        # Calculate correlation between document lengths and agreement scores
        try:
            doc_lengths = [len(metrics['document_metrics'][doc_id].get('human_entities', set()) | 
                              metrics['document_metrics'][doc_id].get('supervisor_entities', set()))
                          for doc_id in metrics['document_metrics']]
            f1_scores = [metrics['document_metrics'][doc_id].get('f1_score', 0) 
                        for doc_id in metrics['document_metrics']]
            
            if doc_lengths and f1_scores and len(doc_lengths) == len(f1_scores) and len(doc_lengths) > 1:
                length_f1_corr, _ = stats.pearsonr(doc_lengths, f1_scores)
                print(f"\nCorrelation between document entity count and F1 score: {length_f1_corr:.4f}")
        except Exception as e:
            print(f"Could not calculate document length correlation: {e}")
    else:
        print("No documents with entities to analyze")

human_doc_entities, supervisor_doc_entities = extract_document_entity_sets(
    human_corrections_full, 
    rdma_corrections
)

# Compute agreement metrics
metrics = compute_agreement_metrics(
    human_doc_entities, 
    supervisor_doc_entities, 
    excluded_docs=None  # Optional: list of document IDs to exclude
    )
print(metrics)

{'total_documents': 82, 'total_unique_entities': 89, 'total_human_entities': 122, 'total_supervisor_entities': 61, 'total_common_entities': 45, 'total_human_only': 77, 'total_supervisor_only': 16, 'overall_precision': 0.7377049180327869, 'overall_recall': 0.36885245901639346, 'overall_f1': 0.49180327868852464, 'cohen_kappa': -0.23760848601735796, 'pearson_correlation': -0.40687454577019194, 'pearson_p_value': 7.33842791751382e-07, 'avg_precision': 0.47560975609756095, 'avg_recall': 0.39105691056910563, 'avg_f1': 0.4187572590011614, 'avg_percent_agreement': 0.39105691056910563, 'std_precision': 0.4994047616937383, 'std_recall': 0.44338097367624796, 'std_f1': 0.4542703666785241, 'std_percent_agreement': 0.44338097367624796, 'document_metrics': {'977': {'human_entities': {'rheumatic fever'}, 'supervisor_entities': set(), 'common_entities': set(), 'human_count': 1, 'supervisor_count': 0, 'common_count': 0, 'human_only_count': 1, 'supervisor_only_count': 0, 'precision': 0, 'recall': 0.0, 'f

In [9]:
print(human_doc_entities)
print(supervisor_doc_entities)

defaultdict(<class 'set'>, {'1208': {'nocardiosis'}, '950': {'retinitis pigmentosa'}, '977': {'rheumatic fever'}, '1552': {'hit'}, '1790': {'tracheobronchomalacia'}, '2452': {'sarcoidosis'}, '3390': {'als'}, '4806': {'rheumatic fever'}, '4938': {'sarcoid'}, '10406': {'medullary sponge kidney'}, '6465': {'hit'}, '13231': {'hyperthyroidism'}, '7688': {'cervical stenosis', 'amyotrophic lateral sclerosis'}, '6188': {'pml', 'hyperthyroidism'}, '8979': {'heparin induced thrombocytopenia', 'sclerosis cholangitis', 'hit'}, '10004': {'asbestosis', 'dilated cardiomyopathy'}, '10715': {'heparin induced thrombocytopenia'}, '9512': {'mediastinitis'}, '8960': {'sick sinus syndrome', 'sarcoid', "bechet's disease"}, '16334': {'cervical stenosis', 'hit'}, '16347': {'hypothyroidism secondary'}, '13666': {'central nervous system and systemic lymphoma'}, '14936': {'bullous pemphigoid', 'methemoglobinemia'}, '11938': {'retinopathy of prematurity'}, '11604': {'antiphospholipid antibody syndrome', 'microcyti

In [12]:
# print(sorted(supervisor_doc_entities.keys()))
for doc_id, entities in human_doc_entities.items():
    print(f"Document ID: {doc_id}")
    print(f"Entities: {entities}")
    print("Supervisor Entities:")
    if doc_id in supervisor_doc_entities:
        print(supervisor_doc_entities[doc_id])
    else:
        print("No supervisor entities found for this document.")
    print()
# print(sorted(human_doc_entities.keys()))

Document ID: 1208
Entities: {'nocardiosis'}
Supervisor Entities:
set()

Document ID: 950
Entities: {'retinitis pigmentosa'}
Supervisor Entities:
{'retinitis pigmentosa'}

Document ID: 977
Entities: {'rheumatic fever'}
Supervisor Entities:
set()

Document ID: 1552
Entities: {'hit'}
Supervisor Entities:
{'heparin-induced thrombocytopenia'}

Document ID: 1790
Entities: {'tracheobronchomalacia'}
Supervisor Entities:
set()

Document ID: 2452
Entities: {'sarcoidosis'}
Supervisor Entities:
{'sarcoidosis'}

Document ID: 3390
Entities: {'als'}
Supervisor Entities:
{'amyotrophic lateral sclerosis'}

Document ID: 4806
Entities: {'rheumatic fever'}
Supervisor Entities:
set()

Document ID: 4938
Entities: {'sarcoid'}
Supervisor Entities:
set()

Document ID: 10406
Entities: {'medullary sponge kidney'}
Supervisor Entities:
{'hit'}

Document ID: 6465
Entities: {'hit'}
Supervisor Entities:
{'hit'}

Document ID: 13231
Entities: {'hyperthyroidism'}
Supervisor Entities:
set()

Document ID: 7688
Entities: {

# All Human vs. RDMA + Human

# 